# House Price Prediction
**Internship Project - Week 1**

**Objective:** Predict house prices using property features and identify the most influential factors affecting price.

## Task 1 - Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_csv('Housing.csv')
df.head(10)

In [ ]:
# Dataset shape
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print('Column names:', list(df.columns))

In [ ]:
# Target and features
target = 'price'
features = [col for col in df.columns if col != target]
print('Target column:', target)
print('Feature columns:', features)

In [ ]:
# Missing values check
df.isnull().sum()

In [ ]:
# Dataset information and descriptive statistics
display(df.info())
df.describe()

## Task 2 - Data Cleaning

In [ ]:
# Make a copy for cleaning
df_clean = df.copy()

# Remove duplicate rows
before_rows = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after_rows = df_clean.shape[0]
print(f'Duplicate rows removed: {before_rows - after_rows}')

# Missing values are checked again
print('Missing values after cleaning:')
print(df_clean.isnull().sum())

In [ ]:
# Convert categorical columns into numeric columns using one-hot encoding
X = df_clean.drop('price', axis=1)
y = df_clean['price']

X_encoded = pd.get_dummies(X, drop_first=True)
print('Original feature count:', X.shape[1])
print('Encoded feature count:', X_encoded.shape[1])
X_encoded.head()

## Task 3 - Model Building and Evaluation

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

In [ ]:
# Train and evaluate models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(
        n_estimators=300, random_state=42, max_depth=None
    )
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Select best model based on R2 Score
best_model_name = results_df.sort_values('R2 Score', ascending=False).iloc[0]['Model']
best_model_name

## Task 4 - Visualizations

In [ ]:
# Chart 1: Distribution of house prices
plt.figure(figsize=(8, 5))
plt.hist(df_clean['price'], bins=25, edgecolor='black')
plt.title('Distribution of House Prices')
plt.xlabel('Price')
plt.ylabel('Number of Houses')
plt.tight_layout()
plt.savefig('charts/price_distribution.png', dpi=200)
plt.show()

In [ ]:
# Chart 2: Correlation heatmap
corr_df = pd.get_dummies(df_clean, drop_first=True)
corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
im = plt.imshow(corr, aspect='auto')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=8)
plt.title('Correlation Heatmap of Housing Features')
plt.tight_layout()
plt.savefig('charts/correlation_heatmap.png', dpi=200)
plt.show()

In [ ]:
# Chart 3: Actual vs Predicted prices for the best model
best_pred = predictions[best_model_name]

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_pred, alpha=0.75)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.title(f'Actual vs Predicted Prices ({best_model_name})')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.tight_layout()
plt.savefig('charts/actual_vs_predicted.png', dpi=200)
plt.show()

In [ ]:
# Bonus chart: Random Forest feature importance
rf_model = models['Random Forest Regressor']
feature_importance = pd.Series(
    rf_model.feature_importances_, index=X_encoded.columns
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
feature_importance.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Feature Importances - Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('charts/feature_importance.png', dpi=200)
plt.show()

feature_importance.head(10)

## Task 5 - Insights & Summary

### Short Summary

The most influential feature in the dataset is **area**, followed by **bathrooms**, **air conditioning**, **parking**, and **stories**. Linear Regression performed slightly better than Random Forest on this split, achieving an R² score of approximately **0.653**, which means it explains about **65.3%** of the variation in house prices. In plain terms, the best model's average error is about **₹9.70 lakh**, so it is useful for rough price estimation but not perfect for final valuation. One surprising point is that Random Forest did not outperform Linear Regression, likely because this small dataset has mostly linear price relationships. For a real estate business, the recommendation is to focus valuation and marketing around property size, bathrooms, air conditioning, parking, and preferred area because these features strongly affect perceived value.


In [ ]:
# Save model comparison and feature importance for reference
results_df.to_csv('model_metrics.csv', index=False)
feature_importance.to_csv('feature_importance.csv', header=['importance'])
print('Saved model_metrics.csv, feature_importance.csv, and chart images in charts/ folder.')